# 06b — Extended Downstream Evaluation with Friend's Pretrained aeon Models

This notebook adapts your friend's `evaluation_classification.ipynb` idea to your project structure.

It supports:

```text
KoVAE:
  rollout_v1
  posterior_bank_v2

TimeVAE:
  prior_v1
```

It can use pretrained real-trained aeon models from:

```text
/home/iailab42/khans1/projects/ir/models/downstream/pretrained/tsai_aeon_results/
```

The pretrained real model is mainly useful for:

```text
train real -> test synthetic
```

because the same real-trained classifier can be reused instead of retraining.

Default:

```text
framework: aeon
modalities: acc, bvp, eda, temp, fused
```

Outputs are saved separately from the pretrained folder:

```text
results/downstream_extended/<model_family>/
figures/downstream_extended/<model_family>/
models/downstream_extended/<model_family>/
```


In [1]:

# ============================================================
# 06b_downstream_activity_evaluation_extended_aeon_pretrained.py
#
# Extended downstream evaluation adapted to this project structure.
#
# Main features:
#   - model-family selection: kovae or timevae
#   - multiple synthetic methods per family
#   - all native modalities: acc, bvp, eda, temp, fused
#   - aeon MiniRocket/Rocket
#   - optional reuse of friend's pretrained real_to_real aeon models
#   - near-flat-window filtering for aeon compatibility
#
# Pretrained external models:
#   models/downstream/pretrained/tsai_aeon_results/
#
# New outputs:
#   results/downstream_extended/<model_family>/
#   figures/downstream_extended/<model_family>/
#   models/downstream_extended/<model_family>/
# ============================================================

from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Optional, Tuple
import inspect
import itertools
import json
import random
import re
import warnings

import joblib
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
)


# ============================================================
# Select model family here
# Options:
#   "kovae"
#   "timevae"
# ============================================================

SELECTED_MODEL_FAMILY = "kovae"
# SELECTED_MODEL_FAMILY = "timevae"


MODEL_FAMILY_CONFIGS = {
    "kovae": {
        "model_family": "kovae",
        "synthetic_base_dir": "data/synthetic_subjects/kovae",
        "synthetic_methods": ["rollout_v1", "posterior_bank_v2"],
        "method_display_names": {
            "rollout_v1": "KoVAE-Rollout",
            "posterior_bank_v2": "KoVAE-Posterior",
        },
    },

    "timevae": {
        "model_family": "timevae",
        "synthetic_base_dir": "data/synthetic_subjects/timevae",
        "synthetic_methods": ["prior_v1"],
        "method_display_names": {
            "prior_v1": "TimeVAE-Prior",
        },
    },
}


# ============================================================
# Config
# ============================================================

EVAL_CONFIG = {
    "project_root": "/home/iailab42/khans1/projects/ir",

    "real_dir": "data/processed/native_rates",

    # Friend's external pretrained results folder.
    # Do not save new results inside this folder.
    "pretrained_results_dir": "models/downstream/pretrained/tsai_aeon_results",

    # New outputs from this adapted notebook.
    "results_base_dir": "results/downstream_extended",
    "figures_base_dir": "figures/downstream_extended",
    "models_base_dir": "models/downstream_extended",
    "configs_dir": "configs",

    # Frameworks supported in this adapted version.
    # Friend's notebook also has tsai; this version focuses on aeon because you received aeon models.
    "frameworks": ["aeon"],

    # Options:
    #   "acc", "bvp", "eda", "temp", "fused"
    "modalities": ["acc", "bvp", "eda", "temp", "fused"],

    # Final evaluation design.
    "include_real_to_synthetic": True,
    "include_synthetic_to_real": True,
    "include_real_plus_synthetic_to_real": True,

    # If True, try to load real_to_real pretrained model for:
    #   real_to_real
    #   real_to_3synthetic
    # If missing, the notebook will train and save a new real_to_real model.
    "use_pretrained_real_models": True,

    # Synthetic test subjects for real -> synthetic.
    "synthetic_test_subject_selection": "first_n",
    "num_synthetic_test_subjects": 3,
    "specific_synthetic_test_subjects": [],

    # Fixed subject split used throughout this project.
    "train_subjects": ["S1", "S2", "S3", "S4", "S5", "S6", "S9", "S11", "S12", "S13"],
    "val_subjects": ["S14", "S15"],
    "test_subjects": ["S7", "S8", "S10"],

    "activity_ids": [1, 2, 3, 4, 5, 6, 7, 8],

    # Fused view resamples all channels to this length.
    "fused_target_len": 512,

    # If runtime is too slow, set this to e.g. 30000.
    # None means use all windows.
    "max_train_windows_per_experiment": None,

    # aeon settings.
    "aeon_n_kernels": 5000,
    "aeon_n_jobs": 8,

    # MiniRocket/Rocket cannot handle case/channel pairs with almost zero std.
    # This only affects temporary classifier arrays.
    "aeon_fix_low_variation": True,
    "aeon_low_variation_strategy": "drop_cases",
    "aeon_min_std": 1e-6,

    "random_seed": 42,

    "save_models": True,
    "save_confusion_matrix_figures": True,
    "show_tables": True,
}


if SELECTED_MODEL_FAMILY not in MODEL_FAMILY_CONFIGS:
    raise ValueError(
        f"Unknown SELECTED_MODEL_FAMILY={SELECTED_MODEL_FAMILY}. "
        f"Available: {list(MODEL_FAMILY_CONFIGS.keys())}"
    )

EVAL_CONFIG.update(MODEL_FAMILY_CONFIGS[SELECTED_MODEL_FAMILY])


# ============================================================
# Constants
# ============================================================

RAW_ARRAY_CONFIGS = {
    "acc": {
        "real_filename": "all_X_acc_32hz.npy",
        "syn_filename": "generated_subjects_X_acc_32hz.npy",
        "expected_shape_tail": (256, 3),
        "native_hz": 32,
        "channel_names": ["ACC_x", "ACC_y", "ACC_z"],
    },
    "bvp": {
        "real_filename": "all_X_bvp_64hz.npy",
        "syn_filename": "generated_subjects_X_bvp_64hz.npy",
        "expected_shape_tail": (512, 1),
        "native_hz": 64,
        "channel_names": ["BVP"],
    },
    "slow": {
        "real_filename": "all_X_slow_4hz.npy",
        "syn_filename": "generated_subjects_X_slow_4hz.npy",
        "expected_shape_tail": (32, 2),
        "native_hz": 4,
        "channel_names": ["EDA", "TEMP"],
    },
}

FUSED_CHANNEL_NAMES = ["ACC_x", "ACC_y", "ACC_z", "BVP", "EDA", "TEMP"]


# ============================================================
# Basic helpers
# ============================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)


def require_file(path: Path) -> Path:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return path


def make_dirs(*dirs: Path) -> None:
    for directory in dirs:
        directory.mkdir(parents=True, exist_ok=True)


def save_json(data: Dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2), encoding="utf-8")


def sanitize_name(value: str) -> str:
    value = str(value)
    value = re.sub(r"[^A-Za-z0-9_.-]+", "_", value)
    value = re.sub(r"_+", "_", value)
    return value.strip("_")


def subject_sort_key(value):
    text = str(value)
    if text.startswith("S") and text[1:].isdigit():
        return (0, int(text[1:]))
    digits = "".join(ch for ch in text if ch.isdigit())
    if digits:
        return (1, int(digits), text)
    return (2, text)


def method_display_name(method_name: str, config: Dict) -> str:
    return config.get("method_display_names", {}).get(method_name, method_name)


def get_paths(config: Dict) -> Dict[str, Path]:
    root = Path(config["project_root"])
    model_family = str(config["model_family"])

    return {
        "root": root,
        "real_dir": root / config["real_dir"],
        "synthetic_base_dir": root / config["synthetic_base_dir"],
        "pretrained_results_dir": root / config["pretrained_results_dir"],
        "results_model_dir": root / config["results_base_dir"] / model_family,
        "figures_model_dir": root / config["figures_base_dir"] / model_family,
        "models_model_dir": root / config["models_base_dir"] / model_family,
        "configs_dir": root / config["configs_dir"],
    }


def safe_display(df: pd.DataFrame, max_rows: int = 50) -> None:
    try:
        display(df)
    except Exception:
        print(df.head(max_rows).to_string(index=False))


def to_channels_first(X_native: np.ndarray) -> np.ndarray:
    X_native = np.asarray(X_native, dtype=np.float32)

    if X_native.ndim != 3:
        raise ValueError(f"Expected [N,T,C], got {X_native.shape}")

    return np.transpose(X_native, (0, 2, 1)).astype(np.float32)


def resample_time_axis(X_native: np.ndarray, target_len: int) -> np.ndarray:
    X_native = np.asarray(X_native, dtype=np.float32)

    if X_native.ndim != 3:
        raise ValueError(f"Expected [N,T,C], got {X_native.shape}")

    n, old_len, channels = X_native.shape
    target_len = int(target_len)

    if old_len == target_len:
        return X_native.copy().astype(np.float32)

    if old_len < 2:
        raise ValueError(f"Cannot resample time axis with old_len={old_len}")

    old_positions = np.linspace(0.0, old_len - 1, target_len, dtype=np.float32)
    left = np.floor(old_positions).astype(np.int64)
    right = np.minimum(left + 1, old_len - 1)
    weight = (old_positions - left).astype(np.float32)

    out = (
        (1.0 - weight)[None, :, None] * X_native[:, left, :]
        + weight[None, :, None] * X_native[:, right, :]
    )

    return out.astype(np.float32)


def split_slow_to_eda_temp(X_slow: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    X_slow = np.asarray(X_slow, dtype=np.float32)

    if X_slow.ndim != 3 or X_slow.shape[2] != 2:
        raise ValueError(f"Expected slow [N,T,2], got {X_slow.shape}")

    return X_slow[:, :, 0:1].astype(np.float32), X_slow[:, :, 1:2].astype(np.float32)


def build_fused_native_view(X_view_dict: Dict[str, np.ndarray], target_len: int) -> np.ndarray:
    X_acc = resample_time_axis(X_view_dict["acc"], target_len)
    X_bvp = resample_time_axis(X_view_dict["bvp"], target_len)
    X_eda = resample_time_axis(X_view_dict["eda"], target_len)
    X_temp = resample_time_axis(X_view_dict["temp"], target_len)

    lengths = {
        "acc": len(X_acc),
        "bvp": len(X_bvp),
        "eda": len(X_eda),
        "temp": len(X_temp),
    }

    if len(set(lengths.values())) != 1:
        raise ValueError(f"Fused view length mismatch: {lengths}")

    return np.concatenate([X_acc, X_bvp, X_eda, X_temp], axis=2).astype(np.float32)


def labels_to_indices(y: np.ndarray, activity_ids: List[int]) -> np.ndarray:
    mapping = {int(label): i for i, label in enumerate(activity_ids)}
    return np.asarray([mapping[int(label)] for label in y], dtype=np.int64)


def indices_to_labels(y_idx: np.ndarray, activity_ids: List[int]) -> np.ndarray:
    activity_ids = [int(x) for x in activity_ids]
    return np.asarray([activity_ids[int(i)] for i in y_idx], dtype=np.int64)


def stratified_subsample(
    X: np.ndarray,
    y: np.ndarray,
    max_samples: Optional[int],
    seed: int,
) -> Tuple[np.ndarray, np.ndarray]:
    if max_samples is None:
        return X, y

    max_samples = int(max_samples)
    if len(y) <= max_samples:
        return X, y

    rng = np.random.default_rng(seed)
    y = np.asarray(y, dtype=np.int64)

    selected = []
    labels = np.unique(y)

    for label in labels:
        idx = np.where(y == label)[0]
        n_take = max(1, int(round(max_samples * len(idx) / len(y))))
        n_take = min(n_take, len(idx))
        selected.append(rng.choice(idx, size=n_take, replace=False))

    selected = np.concatenate(selected)

    if len(selected) > max_samples:
        selected = rng.choice(selected, size=max_samples, replace=False)

    rng.shuffle(selected)

    return X[selected], y[selected]


# ============================================================
# Load data
# ============================================================

def check_raw_array_shape(X: np.ndarray, expected_tail: Tuple[int, int], label: str) -> None:
    if X.ndim != 3 or tuple(X.shape[1:]) != tuple(expected_tail):
        raise ValueError(f"{label}: expected [N,{expected_tail[0]},{expected_tail[1]}], got {X.shape}")


def filter_valid_activities(
    X_dict: Dict[str, np.ndarray],
    y: np.ndarray,
    subjects: np.ndarray,
    activity_ids: List[int],
) -> Tuple[Dict[str, np.ndarray], np.ndarray, np.ndarray]:
    keep = np.isin(y, np.asarray(activity_ids, dtype=np.int64))

    X_out = {key: value[keep].astype(np.float32) for key, value in X_dict.items()}
    y_out = y[keep].astype(np.int64)
    subjects_out = subjects[keep].astype(str)

    return X_out, y_out, subjects_out


def load_real_native_data(real_dir: Path, config: Dict) -> Tuple[Dict[str, np.ndarray], np.ndarray, np.ndarray]:
    y = np.load(require_file(real_dir / "all_y.npy")).astype(np.int64)
    subjects = np.load(require_file(real_dir / "all_subject.npy"), allow_pickle=True).astype(str)

    X = {}

    for key, cfg in RAW_ARRAY_CONFIGS.items():
        arr = np.load(require_file(real_dir / cfg["real_filename"])).astype(np.float32)
        check_raw_array_shape(arr, cfg["expected_shape_tail"], f"real {key}")

        if len(arr) != len(y):
            raise ValueError(f"real {key}/y length mismatch: {len(arr)} vs {len(y)}")

        X[key] = arr

    if len(subjects) != len(y):
        raise ValueError(f"real subjects/y length mismatch: {len(subjects)} vs {len(y)}")

    return filter_valid_activities(X, y, subjects, config["activity_ids"])


def load_synthetic_native_data(synthetic_dir: Path, config: Dict) -> Tuple[Dict[str, np.ndarray], np.ndarray, np.ndarray]:
    y = np.load(require_file(synthetic_dir / "generated_subjects_all_y.npy")).astype(np.int64)
    subjects = np.load(
        require_file(synthetic_dir / "generated_subjects_all_subject.npy"),
        allow_pickle=True,
    ).astype(str)

    X = {}

    for key, cfg in RAW_ARRAY_CONFIGS.items():
        arr = np.load(require_file(synthetic_dir / cfg["syn_filename"])).astype(np.float32)
        check_raw_array_shape(arr, cfg["expected_shape_tail"], f"synthetic {key}")

        if len(arr) != len(y):
            raise ValueError(f"synthetic {key}/y length mismatch: {len(arr)} vs {len(y)}")

        X[key] = arr

    if len(subjects) != len(y):
        raise ValueError(f"synthetic subjects/y length mismatch: {len(subjects)} vs {len(y)}")

    return filter_valid_activities(X, y, subjects, config["activity_ids"])


def filter_by_subjects(
    X_dict: Dict[str, np.ndarray],
    y: np.ndarray,
    subjects: np.ndarray,
    selected_subjects: List[str],
) -> Tuple[Dict[str, np.ndarray], np.ndarray, np.ndarray]:
    selected = set(str(x) for x in selected_subjects)
    keep = np.asarray([str(s) in selected for s in subjects], dtype=bool)

    X_out = {key: value[keep].astype(np.float32) for key, value in X_dict.items()}
    y_out = y[keep].astype(np.int64)
    subjects_out = subjects[keep].astype(str)

    return X_out, y_out, subjects_out


def validate_subject_split(subjects: np.ndarray, config: Dict) -> None:
    available = set(subjects.astype(str))

    missing_train = sorted(set(config["train_subjects"]) - available, key=subject_sort_key)
    missing_val = sorted(set(config["val_subjects"]) - available, key=subject_sort_key)
    missing_test = sorted(set(config["test_subjects"]) - available, key=subject_sort_key)

    if missing_train or missing_val or missing_test:
        raise ValueError(
            f"Missing split subjects. train={missing_train}, val={missing_val}, test={missing_test}"
        )


def prepare_real_splits(
    real_X_all: Dict[str, np.ndarray],
    real_y_all: np.ndarray,
    real_subjects_all: np.ndarray,
    config: Dict,
) -> Dict[str, Tuple[Dict[str, np.ndarray], np.ndarray, np.ndarray]]:
    validate_subject_split(real_subjects_all, config)

    return {
        "train": filter_by_subjects(real_X_all, real_y_all, real_subjects_all, config["train_subjects"]),
        "val": filter_by_subjects(real_X_all, real_y_all, real_subjects_all, config["val_subjects"]),
        "test": filter_by_subjects(real_X_all, real_y_all, real_subjects_all, config["test_subjects"]),
    }


def select_synthetic_test_subjects(subjects: np.ndarray, config: Dict) -> List[str]:
    available = sorted(np.unique(subjects.astype(str)).tolist(), key=subject_sort_key)

    if config["synthetic_test_subject_selection"] == "first_n":
        return available[: int(config["num_synthetic_test_subjects"])]

    if config["synthetic_test_subject_selection"] == "specific":
        selected = [str(x) for x in config["specific_synthetic_test_subjects"]]
        missing = sorted(set(selected) - set(available), key=subject_sort_key)

        if missing:
            raise ValueError(f"Requested synthetic test subjects not found: {missing}")

        return selected

    raise ValueError("synthetic_test_subject_selection must be 'first_n' or 'specific'.")


def build_eval_views(X_raw: Dict[str, np.ndarray], config: Dict) -> Dict[str, np.ndarray]:
    X_eda, X_temp = split_slow_to_eda_temp(X_raw["slow"])

    views = {
        "acc": X_raw["acc"],
        "bvp": X_raw["bvp"],
        "eda": X_eda,
        "temp": X_temp,
    }

    views["fused"] = build_fused_native_view(views, target_len=int(config["fused_target_len"]))

    return views


def get_modality_info(modality: str, config: Dict) -> Dict:
    if modality == "acc":
        return {"native_hz": 32, "channels": "ACC_x,ACC_y,ACC_z", "description": "ACC native 32 Hz"}

    if modality == "bvp":
        return {"native_hz": 64, "channels": "BVP", "description": "BVP native 64 Hz"}

    if modality == "eda":
        return {"native_hz": 4, "channels": "EDA", "description": "EDA native 4 Hz"}

    if modality == "temp":
        return {"native_hz": 4, "channels": "TEMP", "description": "TEMP native 4 Hz"}

    if modality == "fused":
        return {"native_hz": 64, "channels": ",".join(FUSED_CHANNEL_NAMES), "description": "Fused resampled to length 512"}

    raise ValueError(f"Unknown modality: {modality}")


# ============================================================
# aeon low-variation handling
# ============================================================

def keep_nonflat_cases_for_aeon(
    X_channels_first: np.ndarray,
    y: np.ndarray,
    min_std: float = 1e-6,
    verbose: bool = True,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    X = np.asarray(X_channels_first, dtype=np.float32)
    y = np.asarray(y, dtype=np.int64)

    if X.ndim != 3:
        raise ValueError(f"Expected [N,C,T], got {X.shape}")

    std = X.std(axis=2)
    keep_mask = np.all(std > float(min_std), axis=1)

    dropped = int((~keep_mask).sum())

    if keep_mask.sum() == 0:
        raise ValueError("All windows would be dropped by aeon low-variation filter.")

    if verbose and dropped > 0:
        print(
            f"aeon low-variation filter: dropped {dropped}/{len(keep_mask)} windows "
            f"with at least one channel std <= {min_std}"
        )

    return X[keep_mask], y[keep_mask], keep_mask


# ============================================================
# Metrics and saving
# ============================================================

def compute_classification_metrics(y_true: np.ndarray, y_pred: np.ndarray, activity_ids: List[int]) -> Dict[str, float]:
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)

    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_precision": float(precision_score(y_true, y_pred, labels=activity_ids, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, labels=activity_ids, average="macro", zero_division=0)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=activity_ids, average="macro", zero_division=0)),
        "weighted_precision": float(precision_score(y_true, y_pred, labels=activity_ids, average="weighted", zero_division=0)),
        "weighted_recall": float(recall_score(y_true, y_pred, labels=activity_ids, average="weighted", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, labels=activity_ids, average="weighted", zero_division=0)),
        "balanced_accuracy": float(recall_score(y_true, y_pred, labels=activity_ids, average="macro", zero_division=0)),
    }


def compute_per_activity_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    activity_ids: List[int],
) -> pd.DataFrame:
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=activity_ids,
        zero_division=0,
    )

    rows = []

    for i, activity in enumerate(activity_ids):
        activity = int(activity)
        true_mask = y_true == activity
        correct = int(np.sum(true_mask & (y_pred == activity)))
        total = int(np.sum(true_mask))

        rows.append(
            {
                "activity_label": activity,
                "precision": float(precision[i]),
                "recall": float(recall[i]),
                "f1": float(f1[i]),
                "support": int(support[i]),
                "correct_true_activity_windows": correct,
                "total_true_activity_windows": total,
                "true_activity_window_accuracy": float(correct / total) if total > 0 else np.nan,
            }
        )

    return pd.DataFrame(rows)


def plot_confusion_matrix(
    cm: np.ndarray,
    labels: List[int],
    title: str,
    path: Path,
    config: Dict,
) -> None:
    if not bool(config["save_confusion_matrix_figures"]):
        return

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm)
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels([str(x) for x in labels])
    ax.set_yticklabels([str(x) for x in labels])

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            value = int(cm[i, j])
            if value > 0:
                ax.text(j, i, str(value), ha="center", va="center", fontsize=8)

    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)


def save_predictions_cm_activity(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    row_meta: Dict,
    paths: Dict[str, Path],
    config: Dict,
) -> pd.DataFrame:
    framework = row_meta["framework"]
    experiment = row_meta["experiment"]
    method = row_meta["synthetic_method"]
    modality = row_meta["modality"]

    run_key = f"{sanitize_name(framework)}__{sanitize_name(experiment)}__{sanitize_name(method)}__{sanitize_name(modality)}"

    predictions_dir = paths["results_model_dir"] / "predictions"
    cm_dir = paths["results_model_dir"] / "confusion_matrices"
    activity_dir = paths["results_model_dir"] / "per_activity_reports"
    fig_dir = paths["figures_model_dir"] / "confusion_matrices"

    make_dirs(predictions_dir, cm_dir, activity_dir, fig_dir)

    pred_df = pd.DataFrame(
        {
            "y_true": y_true.astype(np.int64),
            "y_pred": y_pred.astype(np.int64),
            "correct": y_true.astype(np.int64) == y_pred.astype(np.int64),
        }
    )
    pred_df.to_csv(predictions_dir / f"{run_key}_predictions.csv", index=False)

    cm = confusion_matrix(y_true, y_pred, labels=config["activity_ids"])
    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{a}" for a in config["activity_ids"]],
        columns=[f"pred_{a}" for a in config["activity_ids"]],
    )
    cm_df.to_csv(cm_dir / f"{run_key}_confusion_matrix.csv")

    plot_confusion_matrix(
        cm=cm,
        labels=config["activity_ids"],
        title=f"{framework} | {experiment} | {method} | {modality}",
        path=fig_dir / f"{run_key}_confusion_matrix.png",
        config=config,
    )

    activity_df = compute_per_activity_metrics(y_true, y_pred, config["activity_ids"])

    for key, value in reversed(row_meta.items()):
        activity_df.insert(0, key, value)

    activity_df.to_csv(activity_dir / f"{run_key}_per_activity_metrics.csv", index=False)

    return activity_df


# ============================================================
# Experiment building
# ============================================================

def build_experiment(
    name: str,
    synthetic_method: str,
    train_X: np.ndarray,
    train_y: np.ndarray,
    test_X: np.ndarray,
    test_y: np.ndarray,
    config: Dict,
) -> Dict:
    train_X, train_y = stratified_subsample(
        train_X,
        train_y,
        max_samples=config["max_train_windows_per_experiment"],
        seed=int(config["random_seed"]) + sum(ord(ch) for ch in str(name + synthetic_method)),
    )

    return {
        "name": name,
        "synthetic_method": synthetic_method,
        "train_X": train_X,
        "train_y": train_y.astype(np.int64),
        "test_X": test_X,
        "test_y": test_y.astype(np.int64),
    }


def build_experiments_for_method_modality(
    method_name: str,
    modality: str,
    real_train_views: Dict[str, np.ndarray],
    real_test_views: Dict[str, np.ndarray],
    real_train_y: np.ndarray,
    real_test_y: np.ndarray,
    syn_views: Dict[str, np.ndarray],
    syn_y: np.ndarray,
    syn_test_views: Dict[str, np.ndarray],
    syn_test_y: np.ndarray,
    config: Dict,
) -> List[Dict]:
    experiments = []

    experiments.append(
        build_experiment(
            name="real_to_real",
            synthetic_method="none",
            train_X=real_train_views[modality],
            train_y=real_train_y,
            test_X=real_test_views[modality],
            test_y=real_test_y,
            config=config,
        )
    )

    if bool(config["include_real_to_synthetic"]):
        experiments.append(
            build_experiment(
                name="real_to_3synthetic",
                synthetic_method=method_name,
                train_X=real_train_views[modality],
                train_y=real_train_y,
                test_X=syn_test_views[modality],
                test_y=syn_test_y,
                config=config,
            )
        )

    if bool(config["include_synthetic_to_real"]):
        experiments.append(
            build_experiment(
                name="synthetic_to_real",
                synthetic_method=method_name,
                train_X=syn_views[modality],
                train_y=syn_y,
                test_X=real_test_views[modality],
                test_y=real_test_y,
                config=config,
            )
        )

    if bool(config["include_real_plus_synthetic_to_real"]):
        experiments.append(
            build_experiment(
                name="real_plus_synthetic_to_real",
                synthetic_method=method_name,
                train_X=np.concatenate([real_train_views[modality], syn_views[modality]], axis=0),
                train_y=np.concatenate([real_train_y, syn_y], axis=0),
                test_X=real_test_views[modality],
                test_y=real_test_y,
                config=config,
            )
        )

    return experiments


# ============================================================
# aeon model loading/saving
# ============================================================

def import_aeon_classifier_class():
    try:
        from aeon.classification.convolution_based import MiniRocketClassifier
        return MiniRocketClassifier
    except Exception:
        try:
            from aeon.classification.convolution_based import RocketClassifier
            return RocketClassifier
        except Exception as exc:
            raise ImportError(f"Could not import MiniRocketClassifier or RocketClassifier: {repr(exc)}")


def make_aeon_classifier(config: Dict):
    Classifier = import_aeon_classifier_class()
    params = inspect.signature(Classifier).parameters
    kwargs = {}

    if "n_kernels" in params:
        kwargs["n_kernels"] = int(config["aeon_n_kernels"])

    if "num_kernels" in params:
        kwargs["num_kernels"] = int(config["aeon_n_kernels"])

    if "n_jobs" in params:
        kwargs["n_jobs"] = int(config["aeon_n_jobs"])

    if "random_state" in params:
        kwargs["random_state"] = int(config["random_seed"])

    return Classifier(**kwargs)


def get_model_save_path(
    paths: Dict[str, Path],
    framework: str,
    model_name: str,
    experiment: str,
    synthetic_method: str,
    modality: str,
) -> Path:
    model_dir = paths["models_model_dir"] / f"{framework}_saved_models"
    model_dir.mkdir(parents=True, exist_ok=True)

    filename = (
        f"{sanitize_name(model_name)}__"
        f"{sanitize_name(experiment)}__"
        f"{sanitize_name(synthetic_method)}__"
        f"{sanitize_name(modality)}.joblib"
    )

    return model_dir / filename


def find_pretrained_real_model(
    modality: str,
    paths: Dict[str, Path],
    config: Dict,
) -> Optional[Path]:
    candidates = []

    # 1. Friend's pretrained model folder.
    pretrained = paths["pretrained_results_dir"]
    if pretrained.exists():
        for sub in ["aeon_saved_models", "saved_models", "models"]:
            model_dir = pretrained / sub
            if model_dir.exists():
                candidates.extend(sorted(model_dir.glob(f"*_real_to_real_{sanitize_name(modality)}.joblib")))
                candidates.extend(sorted(model_dir.glob(f"*real_to_real*{sanitize_name(modality)}*.joblib")))

        candidates.extend(sorted(pretrained.glob(f"**/*real_to_real*{sanitize_name(modality)}*.joblib")))

    # 2. Our adapted notebook output model folder.
    own_model_dir = paths["models_model_dir"] / "aeon_saved_models"
    if own_model_dir.exists():
        candidates.extend(sorted(own_model_dir.glob(f"*real_to_real*none*{sanitize_name(modality)}*.joblib")))
        candidates.extend(sorted(own_model_dir.glob(f"*real_to_real*{sanitize_name(modality)}*.joblib")))

    unique = []
    seen = set()

    for candidate in candidates:
        candidate = Path(candidate)
        if candidate.exists() and str(candidate.resolve()) not in seen:
            unique.append(candidate)
            seen.add(str(candidate.resolve()))

    if unique:
        return unique[0]

    return None


def train_or_load_real_model_for_modality(
    modality: str,
    real_train_X: np.ndarray,
    real_train_y: np.ndarray,
    paths: Dict[str, Path],
    config: Dict,
) -> Tuple[object, str, bool, Dict]:
    pretrained_path = None

    if bool(config["use_pretrained_real_models"]):
        pretrained_path = find_pretrained_real_model(modality, paths, config)

    if pretrained_path is not None:
        print(f"Loading pretrained aeon real_to_real model for {modality}: {pretrained_path}")
        model = joblib.load(pretrained_path)
        return model, str(pretrained_path), True, {
            "real_model_train_windows_original": np.nan,
            "real_model_train_windows_dropped": np.nan,
            "real_model_train_windows_used": np.nan,
        }

    print(f"No pretrained aeon real_to_real model found for {modality}. Training a new one.")

    X_train = to_channels_first(real_train_X)
    y_train = real_train_y.astype(np.int64)
    original_n = len(y_train)

    if bool(config["aeon_fix_low_variation"]):
        X_train, y_train, _ = keep_nonflat_cases_for_aeon(
            X_train,
            y_train,
            min_std=float(config["aeon_min_std"]),
            verbose=True,
        )

    clf = make_aeon_classifier(config)
    clf.fit(X_train, y_train)

    model_path = get_model_save_path(
        paths=paths,
        framework="aeon",
        model_name=clf.__class__.__name__,
        experiment="real_to_real",
        synthetic_method="none",
        modality=modality,
    )
    joblib.dump(clf, model_path)
    print("Saved new real_to_real model:", model_path)

    return clf, str(model_path), False, {
        "real_model_train_windows_original": int(original_n),
        "real_model_train_windows_dropped": int(original_n - len(y_train)),
        "real_model_train_windows_used": int(len(y_train)),
    }


# ============================================================
# aeon run
# ============================================================

def run_aeon_experiment(
    exp: Dict,
    modality: str,
    modality_info: Dict,
    paths: Dict[str, Path],
    config: Dict,
    real_model_cache: Dict[str, Tuple[object, str, bool, Dict]],
    real_train_views: Dict[str, np.ndarray],
    real_train_y: np.ndarray,
) -> Tuple[Dict, pd.DataFrame]:
    print("\n" + "=" * 100)
    print(f"[aeon] {exp['name']} | method={exp['synthetic_method']} | modality={modality}")
    print("=" * 100)

    use_cached_real_model = exp["name"] in {"real_to_real", "real_to_3synthetic"}

    if use_cached_real_model:
        if modality not in real_model_cache:
            real_model_cache[modality] = train_or_load_real_model_for_modality(
                modality=modality,
                real_train_X=real_train_views[modality],
                real_train_y=real_train_y,
                paths=paths,
                config=config,
            )

        clf, saved_model_file, used_pretrained_real_model, real_model_meta = real_model_cache[modality]

        X_test = to_channels_first(exp["test_X"])
        y_test = exp["test_y"].astype(np.int64)
        test_original_n = len(y_test)

        if bool(config["aeon_fix_low_variation"]):
            X_test, y_test, _ = keep_nonflat_cases_for_aeon(
                X_test,
                y_test,
                min_std=float(config["aeon_min_std"]),
                verbose=True,
            )

        train_original_n = int(len(real_train_y))
        train_dropped = real_model_meta.get("real_model_train_windows_dropped", np.nan)
        train_used = real_model_meta.get("real_model_train_windows_used", np.nan)

    else:
        clf = make_aeon_classifier(config)

        X_train = to_channels_first(exp["train_X"])
        X_test = to_channels_first(exp["test_X"])
        y_train = exp["train_y"].astype(np.int64)
        y_test = exp["test_y"].astype(np.int64)

        train_original_n = len(y_train)
        test_original_n = len(y_test)

        if bool(config["aeon_fix_low_variation"]):
            X_train, y_train, _ = keep_nonflat_cases_for_aeon(
                X_train,
                y_train,
                min_std=float(config["aeon_min_std"]),
                verbose=True,
            )
            X_test, y_test, _ = keep_nonflat_cases_for_aeon(
                X_test,
                y_test,
                min_std=float(config["aeon_min_std"]),
                verbose=True,
            )

        print("Aeon model:", clf.__class__.__name__)
        print("Train shape:", X_train.shape, y_train.shape)
        print("Test shape: ", X_test.shape, y_test.shape)

        set_seed(int(config["random_seed"]))
        clf.fit(X_train, y_train)

        saved_model_file = ""
        used_pretrained_real_model = False

        if bool(config["save_models"]):
            model_path = get_model_save_path(
                paths=paths,
                framework="aeon",
                model_name=clf.__class__.__name__,
                experiment=exp["name"],
                synthetic_method=exp["synthetic_method"],
                modality=modality,
            )
            joblib.dump(clf, model_path)
            saved_model_file = str(model_path)
            print("Saved model:", model_path)

        train_dropped = int(train_original_n - len(y_train))
        train_used = int(len(y_train))

    y_pred = clf.predict(X_test).astype(np.int64)
    metrics = compute_classification_metrics(y_test, y_pred, config["activity_ids"])

    model_name = clf.__class__.__name__

    row_meta = {
        "framework": "aeon",
        "model": model_name,
        "model_family": config["model_family"],
        "experiment": exp["name"],
        "synthetic_method": exp["synthetic_method"],
        "synthetic_method_display_name": (
            "none" if exp["synthetic_method"] == "none"
            else method_display_name(exp["synthetic_method"], config)
        ),
        "modality": modality,
    }

    activity_df = save_predictions_cm_activity(
        y_true=y_test,
        y_pred=y_pred,
        row_meta=row_meta,
        paths=paths,
        config=config,
    )

    row = {
        **row_meta,
        "native_hz": int(modality_info["native_hz"]),
        "channels": modality_info["channels"],
        "description": modality_info["description"],
        "train_windows_original_before_aeon_filter": int(train_original_n),
        "train_windows_dropped_by_aeon_filter": (
            int(train_dropped) if pd.notna(train_dropped) else np.nan
        ),
        "train_windows": (
            int(train_used) if pd.notna(train_used) else np.nan
        ),
        "test_windows_original_before_aeon_filter": int(test_original_n),
        "test_windows_dropped_by_aeon_filter": int(test_original_n - len(y_test)),
        "test_windows": int(len(y_test)),
        "used_pretrained_real_model": bool(used_pretrained_real_model),
        "saved_model_file": saved_model_file,
        "pretrained_results_dir": str(paths["pretrained_results_dir"]),
        "aeon_n_kernels": int(config["aeon_n_kernels"]),
        "aeon_n_jobs": int(config["aeon_n_jobs"]),
        "aeon_low_variation_strategy": str(config["aeon_low_variation_strategy"]),
        **metrics,
    }

    print(json.dumps(row, indent=2))

    return row, activity_df


# ============================================================
# Summaries
# ============================================================

def save_split_summary(
    real_splits: Dict[str, Tuple[Dict[str, np.ndarray], np.ndarray, np.ndarray]],
    synthetic_data_by_method: Dict[str, Dict],
    paths: Dict[str, Path],
    config: Dict,
) -> pd.DataFrame:
    rows = []

    for split_name, (_, y, subjects) in real_splits.items():
        row = {
            "dataset": f"real_{split_name}",
            "num_windows": int(len(y)),
            "subjects": ",".join(sorted(np.unique(subjects.astype(str)), key=subject_sort_key)),
        }

        for activity in config["activity_ids"]:
            row[f"activity_{activity}_windows"] = int(np.sum(y == int(activity)))

        rows.append(row)

    for method_name, data in synthetic_data_by_method.items():
        for dataset_name, y, subjects in [
            (f"synthetic_train_all10_{method_name}", data["syn_y"], data["syn_subjects"]),
            (f"synthetic_test_3subjects_{method_name}", data["syn_test_y"], data["syn_test_subjects"]),
        ]:
            row = {
                "dataset": dataset_name,
                "num_windows": int(len(y)),
                "subjects": ",".join(sorted(np.unique(subjects.astype(str)), key=subject_sort_key)),
            }

            for activity in config["activity_ids"]:
                row[f"activity_{activity}_windows"] = int(np.sum(y == int(activity)))

            rows.append(row)

    df = pd.DataFrame(rows)
    out_path = paths["results_model_dir"] / "data_split_summary.csv"
    df.to_csv(out_path, index=False)
    print("Saved:", out_path)

    return df


def save_shape_report(
    real_views: Dict[str, Dict[str, np.ndarray]],
    synthetic_data_by_method: Dict[str, Dict],
    paths: Dict[str, Path],
    config: Dict,
) -> pd.DataFrame:
    rows = []

    for modality in config["modalities"]:
        row = {
            "modality": modality,
            "real_train_shape": list(real_views["train"][modality].shape),
            "real_test_shape": list(real_views["test"][modality].shape),
        }

        for method_name, data in synthetic_data_by_method.items():
            row[f"{method_name}_synthetic_train_shape"] = list(data["syn_views"][modality].shape)
            row[f"{method_name}_synthetic_test_shape"] = list(data["syn_test_views"][modality].shape)

        rows.append(row)

    df = pd.DataFrame(rows)
    out_path = paths["results_model_dir"] / "downstream_extended_shape_report.csv"
    df.to_csv(out_path, index=False)
    print("Saved:", out_path)

    return df


def save_coverage_report(results_df: pd.DataFrame, paths: Dict[str, Path], config: Dict) -> pd.DataFrame:
    expected_rows = []

    for modality in config["modalities"]:
        expected_rows.append(
            {
                "framework": "aeon",
                "experiment": "real_to_real",
                "synthetic_method": "none",
                "modality": modality,
            }
        )

        for method in config["synthetic_methods"]:
            if bool(config["include_real_to_synthetic"]):
                expected_rows.append(
                    {
                        "framework": "aeon",
                        "experiment": "real_to_3synthetic",
                        "synthetic_method": method,
                        "modality": modality,
                    }
                )

            if bool(config["include_synthetic_to_real"]):
                expected_rows.append(
                    {
                        "framework": "aeon",
                        "experiment": "synthetic_to_real",
                        "synthetic_method": method,
                        "modality": modality,
                    }
                )

            if bool(config["include_real_plus_synthetic_to_real"]):
                expected_rows.append(
                    {
                        "framework": "aeon",
                        "experiment": "real_plus_synthetic_to_real",
                        "synthetic_method": method,
                        "modality": modality,
                    }
                )

    expected = pd.DataFrame(expected_rows).drop_duplicates()

    if len(results_df) > 0:
        actual = results_df.copy()

        if "error" not in actual.columns:
            actual["error"] = ""

        actual["has_error"] = actual["error"].fillna("").astype(str) != ""

        grouped = (
            actual.groupby(["framework", "experiment", "synthetic_method", "modality"], dropna=False)
            .agg(row_count=("framework", "size"), error_count=("has_error", "sum"))
            .reset_index()
        )
    else:
        grouped = pd.DataFrame(
            columns=["framework", "experiment", "synthetic_method", "modality", "row_count", "error_count"]
        )

    coverage = expected.merge(
        grouped,
        on=["framework", "experiment", "synthetic_method", "modality"],
        how="left",
    )

    coverage["row_count"] = coverage["row_count"].fillna(0).astype(int)
    coverage["error_count"] = coverage["error_count"].fillna(0).astype(int)
    coverage["status"] = np.where(
        coverage["row_count"] == 0,
        "missing",
        np.where(coverage["error_count"] > 0, "error", "ok"),
    )

    out_path = paths["results_model_dir"] / "experiment_coverage_report.csv"
    coverage.to_csv(out_path, index=False)
    print("Saved:", out_path)

    return coverage


def save_compact_tables(results_df: pd.DataFrame, activity_df: pd.DataFrame, paths: Dict[str, Path]) -> Dict[str, Path]:
    output_paths = {}

    aggregate_path = paths["results_model_dir"] / "all_framework_native_results.csv"
    results_df.to_csv(aggregate_path, index=False)
    output_paths["aggregate_results"] = aggregate_path
    print("Saved:", aggregate_path)

    activity_path = paths["results_model_dir"] / "all_framework_native_per_activity_results.csv"
    activity_df.to_csv(activity_path, index=False)
    output_paths["per_activity_results"] = activity_path
    print("Saved:", activity_path)

    if len(results_df) > 0 and "macro_f1" in results_df.columns:
        valid = results_df.dropna(subset=["macro_f1"]).copy()

        cols = [
            "framework",
            "model_family",
            "modality",
            "experiment",
            "synthetic_method",
            "synthetic_method_display_name",
            "accuracy",
            "macro_f1",
            "weighted_f1",
            "balanced_accuracy",
            "used_pretrained_real_model",
            "train_windows_dropped_by_aeon_filter",
            "test_windows_dropped_by_aeon_filter",
        ]
        cols = [c for c in cols if c in valid.columns]

        compact = valid[cols].copy()
        compact_path = paths["results_model_dir"] / "compact_downstream_extended_comparison.csv"
        compact.to_csv(compact_path, index=False)
        output_paths["compact_comparison"] = compact_path
        print("Saved:", compact_path)

        ranking = valid.sort_values(
            by=["modality", "macro_f1"],
            ascending=[True, False],
        )
        ranking_path = paths["results_model_dir"] / "downstream_extended_macro_f1_ranking.csv"
        ranking.to_csv(ranking_path, index=False)
        output_paths["macro_f1_ranking"] = ranking_path
        print("Saved:", ranking_path)

    return output_paths


# ============================================================
# Main runner
# ============================================================

def main(config: Dict = EVAL_CONFIG) -> Dict[str, object]:
    set_seed(int(config["random_seed"]))

    paths = get_paths(config)

    make_dirs(
        paths["results_model_dir"],
        paths["figures_model_dir"],
        paths["models_model_dir"],
        paths["configs_dir"],
    )

    config_path = paths["configs_dir"] / f"downstream_extended_{config['model_family']}_config.json"
    save_json(config, config_path)

    print("=" * 100)
    print("06b Extended downstream evaluation")
    print("=" * 100)
    print("Selected model family:", config["model_family"])
    print("Synthetic methods:", config["synthetic_methods"])
    print("Modalities:", config["modalities"])
    print("Use pretrained real models:", config["use_pretrained_real_models"])
    print("Pretrained folder:", paths["pretrained_results_dir"])
    print("Results folder:", paths["results_model_dir"])
    print("Figures folder:", paths["figures_model_dir"])
    print("Models folder:", paths["models_model_dir"])
    print("Config saved:", config_path)

    real_X_all, real_y_all, real_subjects_all = load_real_native_data(paths["real_dir"], config)
    real_splits = prepare_real_splits(real_X_all, real_y_all, real_subjects_all, config)

    real_train_X_raw, real_train_y, real_train_subjects = real_splits["train"]
    real_test_X_raw, real_test_y, real_test_subjects = real_splits["test"]

    real_views = {
        "train": build_eval_views(real_train_X_raw, config),
        "test": build_eval_views(real_test_X_raw, config),
    }

    synthetic_data_by_method = {}

    for method_name in config["synthetic_methods"]:
        synthetic_dir = paths["synthetic_base_dir"] / method_name

        print("\n" + "#" * 100)
        print(f"Loading synthetic method: {method_name} ({method_display_name(method_name, config)})")
        print("#" * 100)
        print("Synthetic folder:", synthetic_dir)

        syn_X_raw, syn_y, syn_subjects = load_synthetic_native_data(synthetic_dir, config)
        syn_views = build_eval_views(syn_X_raw, config)

        selected_syn_test_subjects = select_synthetic_test_subjects(syn_subjects, config)
        print("Selected synthetic test subjects:", selected_syn_test_subjects)

        syn_test_X_raw, syn_test_y, syn_test_subjects = filter_by_subjects(
            X_dict=syn_X_raw,
            y=syn_y,
            subjects=syn_subjects,
            selected_subjects=selected_syn_test_subjects,
        )
        syn_test_views = build_eval_views(syn_test_X_raw, config)

        synthetic_data_by_method[method_name] = {
            "syn_X_raw": syn_X_raw,
            "syn_y": syn_y,
            "syn_subjects": syn_subjects,
            "syn_views": syn_views,
            "syn_test_X_raw": syn_test_X_raw,
            "syn_test_y": syn_test_y,
            "syn_test_subjects": syn_test_subjects,
            "syn_test_views": syn_test_views,
        }

    split_summary_df = save_split_summary(real_splits, synthetic_data_by_method, paths, config)
    shape_report_df = save_shape_report(real_views, synthetic_data_by_method, paths, config)

    if bool(config["show_tables"]):
        print("\nSplit summary:")
        safe_display(split_summary_df)
        print("\nShape report:")
        safe_display(shape_report_df)

    all_rows = []
    activity_tables = []
    real_model_cache = {}

    for modality in config["modalities"]:
        modality_info = get_modality_info(modality, config)

        # Run real_to_real once per modality.
        real_exp = build_experiment(
            name="real_to_real",
            synthetic_method="none",
            train_X=real_views["train"][modality],
            train_y=real_train_y,
            test_X=real_views["test"][modality],
            test_y=real_test_y,
            config=config,
        )

        try:
            row, activity_df = run_aeon_experiment(
                exp=real_exp,
                modality=modality,
                modality_info=modality_info,
                paths=paths,
                config=config,
                real_model_cache=real_model_cache,
                real_train_views=real_views["train"],
                real_train_y=real_train_y,
            )
            all_rows.append(row)
            activity_tables.append(activity_df)
        except Exception as exc:
            warnings.warn(f"[aeon] failed real_to_real | {modality}: {repr(exc)}")
            all_rows.append(
                {
                    "framework": "aeon",
                    "model_family": config["model_family"],
                    "experiment": "real_to_real",
                    "synthetic_method": "none",
                    "modality": modality,
                    "error": repr(exc),
                }
            )

        # Run method-specific synthetic experiments.
        for method_name, data in synthetic_data_by_method.items():
            method_experiments = build_experiments_for_method_modality(
                method_name=method_name,
                modality=modality,
                real_train_views=real_views["train"],
                real_test_views=real_views["test"],
                real_train_y=real_train_y,
                real_test_y=real_test_y,
                syn_views=data["syn_views"],
                syn_y=data["syn_y"],
                syn_test_views=data["syn_test_views"],
                syn_test_y=data["syn_test_y"],
                config=config,
            )

            for exp in method_experiments:
                # real_to_real was already run above.
                if exp["name"] == "real_to_real":
                    continue

                try:
                    row, activity_df = run_aeon_experiment(
                        exp=exp,
                        modality=modality,
                        modality_info=modality_info,
                        paths=paths,
                        config=config,
                        real_model_cache=real_model_cache,
                        real_train_views=real_views["train"],
                        real_train_y=real_train_y,
                    )
                    all_rows.append(row)
                    activity_tables.append(activity_df)

                except Exception as exc:
                    warnings.warn(
                        f"[aeon] failed {exp['name']} | method={method_name} | modality={modality}: {repr(exc)}"
                    )
                    all_rows.append(
                        {
                            "framework": "aeon",
                            "model_family": config["model_family"],
                            "experiment": exp["name"],
                            "synthetic_method": method_name,
                            "synthetic_method_display_name": method_display_name(method_name, config),
                            "modality": modality,
                            "error": repr(exc),
                        }
                    )

    results_df = pd.DataFrame(all_rows)
    activity_df = pd.concat(activity_tables, ignore_index=True) if activity_tables else pd.DataFrame()

    output_paths = save_compact_tables(results_df, activity_df, paths)
    coverage_df = save_coverage_report(results_df, paths, config)

    summary = {
        "model_family": config["model_family"],
        "synthetic_methods": config["synthetic_methods"],
        "method_display_names": config["method_display_names"],
        "modalities": config["modalities"],
        "frameworks": config["frameworks"],
        "pretrained_results_dir": str(paths["pretrained_results_dir"]),
        "results_model_dir": str(paths["results_model_dir"]),
        "figures_model_dir": str(paths["figures_model_dir"]),
        "models_model_dir": str(paths["models_model_dir"]),
        "config_path": str(config_path),
        "output_paths": {key: str(value) for key, value in output_paths.items()},
        "main_difference_from_notebook_06": (
            "This extended version runs all modalities and can reuse pretrained real-to-real aeon models "
            "for real-to-synthetic evaluation."
        ),
        "evaluation_design": [
            "train real -> test real",
            "train real -> test 3 synthetic subjects",
            "train synthetic -> test real",
            "train real + synthetic -> test real",
        ],
    }

    summary_path = paths["results_model_dir"] / "downstream_extended_evaluation_summary.json"
    save_json(summary, summary_path)
    print("Saved:", summary_path)

    print("\n" + "=" * 100)
    print("DONE")
    print("=" * 100)

    if bool(config["show_tables"]):
        print("\nAggregate results:")
        safe_display(results_df)
        print("\nCoverage:")
        safe_display(coverage_df)

    return {
        "paths": paths,
        "split_summary_df": split_summary_df,
        "shape_report_df": shape_report_df,
        "results_df": results_df,
        "activity_df": activity_df,
        "coverage_df": coverage_df,
        "output_paths": output_paths,
        "summary_path": summary_path,
    }


if __name__ == "__main__":
    outputs = main(EVAL_CONFIG)


06b Extended downstream evaluation
Selected model family: kovae
Synthetic methods: ['rollout_v1', 'posterior_bank_v2']
Modalities: ['acc', 'bvp', 'eda', 'temp', 'fused']
Use pretrained real models: True
Pretrained folder: /home/iailab42/khans1/projects/ir/models/downstream/pretrained/tsai_aeon_results
Results folder: /home/iailab42/khans1/projects/ir/results/downstream_extended/kovae
Figures folder: /home/iailab42/khans1/projects/ir/figures/downstream_extended/kovae
Models folder: /home/iailab42/khans1/projects/ir/models/downstream_extended/kovae
Config saved: /home/iailab42/khans1/projects/ir/configs/downstream_extended_kovae_config.json

####################################################################################################
Loading synthetic method: rollout_v1 (KoVAE-Rollout)
####################################################################################################
Synthetic folder: /home/iailab42/khans1/projects/ir/data/synthetic_subjects/kovae/rollout_v1
Sele

,dataset,num_windows,subjects,activity_1_windows,activity_2_windows,activity_3_windows,activity_4_windows,activity_5_windows,activity_6_windows,activity_7_windows,activity_8_windows
0,real_train,30762,"S1,S2,S3,S4,S5,S6,S9,S11,S12,S13",3032,2139,1500,2296,4580,8792,2903,5520
1,real_val,6100,"S14,S15",605,432,335,449,866,1581,632,1200
2,real_test,10063,"S7,S8,S10",901,635,444,697,1363,3147,1128,1748
3,synthetic_train_all10_rollout_v1,30000,"synthetic_subject_01,synthetic_subject_02,synt...",2900,2051,1397,2190,4616,8628,2756,5462
4,synthetic_test_3subjects_rollout_v1,9000,"synthetic_subject_01,synthetic_subject_02,synt...",898,589,417,677,1437,2488,862,1632
5,synthetic_train_all10_posterior_bank_v2,30000,"synthetic_subject_01,synthetic_subject_02,synt...",2951,2108,1443,2204,4400,8778,2779,5337
6,synthetic_test_3subjects_posterior_bank_v2,9000,"synthetic_subject_01,synthetic_subject_02,synt...",851,624,420,676,1311,2694,863,1561



Shape report:


,modality,real_train_shape,real_test_shape,rollout_v1_synthetic_train_shape,rollout_v1_synthetic_test_shape,posterior_bank_v2_synthetic_train_shape,posterior_bank_v2_synthetic_test_shape
0,acc,"[30762, 256, 3]","[10063, 256, 3]","[30000, 256, 3]","[9000, 256, 3]","[30000, 256, 3]","[9000, 256, 3]"
1,bvp,"[30762, 512, 1]","[10063, 512, 1]","[30000, 512, 1]","[9000, 512, 1]","[30000, 512, 1]","[9000, 512, 1]"
2,eda,"[30762, 32, 1]","[10063, 32, 1]","[30000, 32, 1]","[9000, 32, 1]","[30000, 32, 1]","[9000, 32, 1]"
3,temp,"[30762, 32, 1]","[10063, 32, 1]","[30000, 32, 1]","[9000, 32, 1]","[30000, 32, 1]","[9000, 32, 1]"
4,fused,"[30762, 512, 6]","[10063, 512, 6]","[30000, 512, 6]","[9000, 512, 6]","[30000, 512, 6]","[9000, 512, 6]"



[aeon] real_to_real | method=none | modality=acc
Loading pretrained aeon real_to_real model for acc: /home/iailab42/khans1/projects/ir/models/downstream/pretrained/tsai_aeon_results/aeon_saved_models/MiniRocketClassifier_real_to_real_acc.joblib
aeon low-variation filter: dropped 105/10063 windows with at least one channel std <= 1e-06


/home/iai/user/khans1/.local/lib/python3.12/site-packages/numba/np/ufunc/parallel.py:373: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


{
  "framework": "aeon",
  "model": "MiniRocketClassifier",
  "model_family": "kovae",
  "experiment": "real_to_real",
  "synthetic_method": "none",
  "synthetic_method_display_name": "none",
  "modality": "acc",
  "native_hz": 32,
  "channels": "ACC_x,ACC_y,ACC_z",
  "description": "ACC native 32 Hz",
  "train_windows_original_before_aeon_filter": 30762,
  "train_windows_dropped_by_aeon_filter": NaN,
  "train_windows": NaN,
  "test_windows_original_before_aeon_filter": 10063,
  "test_windows_dropped_by_aeon_filter": 105,
  "test_windows": 9958,
  "used_pretrained_real_model": true,
  "saved_model_file": "/home/iailab42/khans1/projects/ir/models/downstream/pretrained/tsai_aeon_results/aeon_saved_models/MiniRocketClassifier_real_to_real_acc.joblib",
  "pretrained_results_dir": "/home/iailab42/khans1/projects/ir/models/downstream/pretrained/tsai_aeon_results",
  "aeon_n_kernels": 5000,
  "aeon_n_jobs": 8,
  "aeon_low_variation_strategy": "drop_cases",
  "accuracy": 0.6878891343643302,
  

,framework,model,model_family,experiment,synthetic_method,synthetic_method_display_name,modality,native_hz,channels,description,...,aeon_n_jobs,aeon_low_variation_strategy,accuracy,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1,balanced_accuracy
0,aeon,MiniRocketClassifier,kovae,real_to_real,none,none,acc,32,"ACC_x,ACC_y,ACC_z",ACC native 32 Hz,...,8,drop_cases,0.687889,0.749033,0.715609,0.726838,0.705445,0.687889,0.692361,0.715609
1,aeon,MiniRocketClassifier,kovae,real_to_3synthetic,rollout_v1,KoVAE-Rollout,acc,32,"ACC_x,ACC_y,ACC_z",ACC native 32 Hz,...,8,drop_cases,0.277222,0.174490,0.127204,0.060310,0.174123,0.277222,0.125366,0.127204
2,aeon,MiniRocketClassifier,kovae,synthetic_to_real,rollout_v1,KoVAE-Rollout,acc,32,"ACC_x,ACC_y,ACC_z",ACC native 32 Hz,...,8,drop_cases,0.227656,0.432155,0.384122,0.255474,0.454210,0.227656,0.175632,0.384122
3,aeon,MiniRocketClassifier,kovae,real_plus_synthetic_to_real,rollout_v1,KoVAE-Rollout,acc,32,"ACC_x,ACC_y,ACC_z",ACC native 32 Hz,...,8,drop_cases,0.700643,0.751766,0.713643,0.728527,0.712743,0.700643,0.703817,0.713643
4,aeon,MiniRocketClassifier,kovae,real_to_3synthetic,posterior_bank_v2,KoVAE-Posterior,acc,32,"ACC_x,ACC_y,ACC_z",ACC native 32 Hz,...,8,drop_cases,0.672333,0.853098,0.674132,0.705519,0.764061,0.672333,0.642747,0.674132
5,aeon,MiniRocketClassifier,kovae,synthetic_to_real,posterior_bank_v2,KoVAE-Posterior,acc,32,"ACC_x,ACC_y,ACC_z",ACC native 32 Hz,...,8,drop_cases,0.654650,0.687816,0.697992,0.687505,0.656646,0.654650,0.650930,0.697992
6,aeon,MiniRocketClassifier,kovae,real_plus_synthetic_to_real,posterior_bank_v2,KoVAE-Posterior,acc,32,"ACC_x,ACC_y,ACC_z",ACC native 32 Hz,...,8,drop_cases,0.703856,0.765083,0.725967,0.738855,0.720828,0.703856,0.707425,0.725967
7,aeon,MiniRocketClassifier,kovae,real_to_real,none,none,bvp,64,BVP,BVP native 64 Hz,...,8,drop_cases,0.450661,0.495461,0.465159,0.465796,0.463723,0.450661,0.446103,0.465159
8,aeon,MiniRocketClassifier,kovae,real_to_3synthetic,rollout_v1,KoVAE-Rollout,bvp,64,BVP,BVP native 64 Hz,...,8,drop_cases,0.178556,0.172351,0.140157,0.099915,0.200680,0.178556,0.149129,0.140157
9,aeon,MiniRocketClassifier,kovae,synthetic_to_real,rollout_v1,KoVAE-Rollout,bvp,64,BVP,BVP native 64 Hz,...,8,drop_cases,0.159197,0.231806,0.170444,0.102215,0.175015,0.159197,0.080265,0.170444



Coverage:


,framework,experiment,synthetic_method,modality,row_count,error_count,status
0,aeon,real_to_real,none,acc,1,0,ok
1,aeon,real_to_3synthetic,rollout_v1,acc,1,0,ok
2,aeon,synthetic_to_real,rollout_v1,acc,1,0,ok
3,aeon,real_plus_synthetic_to_real,rollout_v1,acc,1,0,ok
4,aeon,real_to_3synthetic,posterior_bank_v2,acc,1,0,ok
5,aeon,synthetic_to_real,posterior_bank_v2,acc,1,0,ok
6,aeon,real_plus_synthetic_to_real,posterior_bank_v2,acc,1,0,ok
7,aeon,real_to_real,none,bvp,1,0,ok
8,aeon,real_to_3synthetic,rollout_v1,bvp,1,0,ok
9,aeon,synthetic_to_real,rollout_v1,bvp,1,0,ok


## Final run

Before running, put your friend's folder here:

```text
/home/iailab42/khans1/projects/ir/models/downstream/pretrained/tsai_aeon_results/
```

Default:

```python
SELECTED_MODEL_FAMILY = "kovae"
```

To evaluate TimeVAE instead, change:

```python
SELECTED_MODEL_FAMILY = "timevae"
```

The notebook will save new results under:

```text
results/downstream_extended/<model_family>/
```


In [ ]:
# ============================================================
# Run configuration
# ============================================================

EVAL_CONFIG["project_root"] = "/home/iailab42/khans1/projects/ir"

# Pretrained friend folder.
EVAL_CONFIG["pretrained_results_dir"] = "models/downstream/pretrained/tsai_aeon_results"

# Recommended default for extended evaluation.
EVAL_CONFIG["frameworks"] = ["aeon"]
EVAL_CONFIG["modalities"] = ["acc", "bvp", "eda", "temp", "fused"]

# Use friend's pretrained real-to-real aeon models if available.
EVAL_CONFIG["use_pretrained_real_models"] = True

# Exact synthetic-test design.
EVAL_CONFIG["include_real_to_synthetic"] = True
EVAL_CONFIG["include_synthetic_to_real"] = True
EVAL_CONFIG["include_real_plus_synthetic_to_real"] = True
EVAL_CONFIG["synthetic_test_subject_selection"] = "first_n"
EVAL_CONFIG["num_synthetic_test_subjects"] = 3
EVAL_CONFIG["specific_synthetic_test_subjects"] = []

# aeon settings.
EVAL_CONFIG["aeon_n_kernels"] = 5000
EVAL_CONFIG["aeon_n_jobs"] = 8

# Keep aeon flat-window fix.
EVAL_CONFIG["aeon_fix_low_variation"] = True
EVAL_CONFIG["aeon_low_variation_strategy"] = "drop_cases"
EVAL_CONFIG["aeon_min_std"] = 1e-6

# None means use all windows.
EVAL_CONFIG["max_train_windows_per_experiment"] = None

outputs = main(EVAL_CONFIG)
outputs["results_df"]


06b Extended downstream evaluation
Selected model family: kovae
Synthetic methods: ['rollout_v1', 'posterior_bank_v2']
Modalities: ['acc', 'bvp', 'eda', 'temp', 'fused']
Use pretrained real models: True
Pretrained folder: /home/iailab42/khans1/projects/ir/models/downstream/pretrained/tsai_aeon_results
Results folder: /home/iailab42/khans1/projects/ir/results/downstream_extended/kovae
Figures folder: /home/iailab42/khans1/projects/ir/figures/downstream_extended/kovae
Models folder: /home/iailab42/khans1/projects/ir/models/downstream_extended/kovae
Config saved: /home/iailab42/khans1/projects/ir/configs/downstream_extended_kovae_config.json

####################################################################################################
Loading synthetic method: rollout_v1 (KoVAE-Rollout)
####################################################################################################
Synthetic folder: /home/iailab42/khans1/projects/ir/data/synthetic_subjects/kovae/rollout_v1
Sele

,dataset,num_windows,subjects,activity_1_windows,activity_2_windows,activity_3_windows,activity_4_windows,activity_5_windows,activity_6_windows,activity_7_windows,activity_8_windows
0,real_train,30762,"S1,S2,S3,S4,S5,S6,S9,S11,S12,S13",3032,2139,1500,2296,4580,8792,2903,5520
1,real_val,6100,"S14,S15",605,432,335,449,866,1581,632,1200
2,real_test,10063,"S7,S8,S10",901,635,444,697,1363,3147,1128,1748
3,synthetic_train_all10_rollout_v1,30000,"synthetic_subject_01,synthetic_subject_02,synt...",2900,2051,1397,2190,4616,8628,2756,5462
4,synthetic_test_3subjects_rollout_v1,9000,"synthetic_subject_01,synthetic_subject_02,synt...",898,589,417,677,1437,2488,862,1632
5,synthetic_train_all10_posterior_bank_v2,30000,"synthetic_subject_01,synthetic_subject_02,synt...",2951,2108,1443,2204,4400,8778,2779,5337
6,synthetic_test_3subjects_posterior_bank_v2,9000,"synthetic_subject_01,synthetic_subject_02,synt...",851,624,420,676,1311,2694,863,1561



Shape report:


,modality,real_train_shape,real_test_shape,rollout_v1_synthetic_train_shape,rollout_v1_synthetic_test_shape,posterior_bank_v2_synthetic_train_shape,posterior_bank_v2_synthetic_test_shape
0,acc,"[30762, 256, 3]","[10063, 256, 3]","[30000, 256, 3]","[9000, 256, 3]","[30000, 256, 3]","[9000, 256, 3]"
1,bvp,"[30762, 512, 1]","[10063, 512, 1]","[30000, 512, 1]","[9000, 512, 1]","[30000, 512, 1]","[9000, 512, 1]"
2,eda,"[30762, 32, 1]","[10063, 32, 1]","[30000, 32, 1]","[9000, 32, 1]","[30000, 32, 1]","[9000, 32, 1]"
3,temp,"[30762, 32, 1]","[10063, 32, 1]","[30000, 32, 1]","[9000, 32, 1]","[30000, 32, 1]","[9000, 32, 1]"
4,fused,"[30762, 512, 6]","[10063, 512, 6]","[30000, 512, 6]","[9000, 512, 6]","[30000, 512, 6]","[9000, 512, 6]"



[aeon] real_to_real | method=none | modality=acc
Loading pretrained aeon real_to_real model for acc: /home/iailab42/khans1/projects/ir/models/downstream/pretrained/tsai_aeon_results/aeon_saved_models/MiniRocketClassifier_real_to_real_acc.joblib
aeon low-variation filter: dropped 105/10063 windows with at least one channel std <= 1e-06
{
  "framework": "aeon",
  "model": "MiniRocketClassifier",
  "model_family": "kovae",
  "experiment": "real_to_real",
  "synthetic_method": "none",
  "synthetic_method_display_name": "none",
  "modality": "acc",
  "native_hz": 32,
  "channels": "ACC_x,ACC_y,ACC_z",
  "description": "ACC native 32 Hz",
  "train_windows_original_before_aeon_filter": 30762,
  "train_windows_dropped_by_aeon_filter": NaN,
  "train_windows": NaN,
  "test_windows_original_before_aeon_filter": 10063,
  "test_windows_dropped_by_aeon_filter": 105,
  "test_windows": 9958,
  "used_pretrained_real_model": true,
  "saved_model_file": "/home/iailab42/khans1/projects/ir/models/downstre